In [3]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURES = ["unit_weight_kNm3", "cohesion_kPa", "friction_angle_deg",
            "slope_angle_deg", "slope_height_m", "pore_pressure_ratio"]
TARGET = "Status"

# --- 1. Load the 80/20 split -------------------------------------------------
train = pd.read_csv("train80_split.csv")
test = pd.read_csv("test_split.csv")

def xy(df):
    X = df[FEATURES].values
    y = (df[TARGET] == "Unstable").astype(int).values  # 1 = Unstable, 0 = Stable
    return X, y

X_train, y_train = xy(train)
X_test, y_test = xy(test)

print(f"Train (80%): {len(X_train)}, Test (20%): {len(X_test)}")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)

#  2. xgb.cv() over a small hyperparameter grid, picking rounds via ------
#        early stopping WITHIN each fold.
# XGBoost does NOT require feature scaling (it's tree-based) - no scaler
# needed here, same as before.
param_grid = [
    {"max_depth": d, "learning_rate": lr, "subsample": 0.8, "colsample_bytree": 0.8,
     "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42}
    for d in [2, 3, 4]
    for lr in [0.01, 0.05, 0.1]
]

best_logloss = np.inf
best_params = None
best_rounds = None

for params in param_grid:
    cv_results = xgb.cv(
        params, dtrain,
        num_boost_round=500,
        nfold=5,
        stratified=True,
        early_stopping_rounds=20,
        metrics="logloss",
        as_pandas=True,
        seed=42,
        verbose_eval=False,
    )
    mean_logloss = cv_results["test-logloss-mean"].min()
    n_rounds = int(cv_results["test-logloss-mean"].idxmin()) + 1
    if mean_logloss < best_logloss:
        best_logloss = mean_logloss
        best_params = params
        best_rounds = n_rounds

print(f"\nBest hyperparameters (via 5-fold CV on the 80% training pool): "
      f"max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}")
print(f"Best number of boosting rounds (from CV, with early stopping per fold): {best_rounds}")
print(f"Best CV log-loss: {best_logloss:.4f}")

# --- 3. Fit the FINAL model on the FULL 80% training pool -------------------
#        (no data held back - this is the point of moving to 80/20 + CV)
model = xgb.XGBClassifier(
    n_estimators=best_rounds,
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train)

# --- 4. Evaluate ONCE on the held-out TEST set -------------------------------
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

print(f"\nFinal held-out TEST accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Final held-out TEST ROC-AUC: {roc_auc_score(y_test, pred_proba):.3f}")
print("\nClassification report (test set):")
print(classification_report(y_test, pred, target_names=["Stable", "Unstable"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, pred))

# --- 5. Feature importance ---------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
importances.plot.barh(ax=ax, color="#2a6f97")
ax.set_xlabel("XGBoost feature importance (gain-based)")
ax.set_title("XGBoost - key instability indicators\n(80% training pool, 5-fold CV-selected hyperparameters)")
fig.tight_layout()
fig.savefig("xgboost_feature_importance.png", dpi=150)
print("\nSaved xgboost_feature_importance.png")

# --- 6. Save the trained model for reuse -------------------------------------
model.save_model("xgboost_slope_model.json")
print("Saved trained model -> xgboost_slope_model.json")

Train (80%): 443, Test (20%): 111

Best hyperparameters (via 5-fold CV on the 80% training pool): max_depth=4, learning_rate=0.1
Best number of boosting rounds (from CV, with early stopping per fold): 82
Best CV log-loss: 0.3608

Final held-out TEST accuracy: 0.838
Final held-out TEST ROC-AUC: 0.912

Classification report (test set):
              precision    recall  f1-score   support

      Stable       0.82      0.85      0.84        54
    Unstable       0.85      0.82      0.84        57

    accuracy                           0.84       111
   macro avg       0.84      0.84      0.84       111
weighted avg       0.84      0.84      0.84       111

Confusion matrix (rows=actual, cols=predicted):
[[46  8]
 [10 47]]

Saved xgboost_feature_importance.png
Saved trained model -> xgboost_slope_model.json


In [1]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURES = ["unit_weight_kNm3", "cohesion_kPa", "friction_angle_deg",
            "slope_angle_deg", "slope_height_m", "pore_pressure_ratio"]
TARGET = "Status"

# --- 1. Load the 80/20 split -------------------------------------------------
train = pd.read_csv("train80_split.csv")
test = pd.read_csv("test_split.csv")

def xy(df):
    X = df[FEATURES].values
    y = (df[TARGET] == "Unstable").astype(int).values  # 1 = Unstable, 0 = Stable
    return X, y

X_train, y_train = xy(train)
X_test, y_test = xy(test)

print(f"Train (80%): {len(X_train)}, Test (20%): {len(X_test)}")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)

# --- 2. xgb.cv() over a small hyperparameter grid, picking rounds via ------
#        early stopping WITHIN each fold.
# XGBoost does NOT require feature scaling (it's tree-based) - no scaler
# needed here, same as before.
param_grid = [
    {"max_depth": d, "learning_rate": lr, "subsample": 0.8, "colsample_bytree": 0.8,
     "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42}
    for d in [2, 3, 4]
    for lr in [0.01, 0.05, 0.1]
]

best_logloss = np.inf
best_params = None
best_rounds = None

for params in param_grid:
    cv_results = xgb.cv(
        params, dtrain,
        num_boost_round=500,
        nfold=5,
        stratified=True,
        early_stopping_rounds=20,
        metrics="logloss",
        as_pandas=True,
        seed=42,
        verbose_eval=False,
    )
    mean_logloss = cv_results["test-logloss-mean"].min()
    n_rounds = int(cv_results["test-logloss-mean"].idxmin()) + 1
    if mean_logloss < best_logloss:
        best_logloss = mean_logloss
        best_params = params
        best_rounds = n_rounds

print(f"\nBest hyperparameters (via 5-fold CV on the 80% training pool): "
      f"max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}")
print(f"Best number of boosting rounds (from CV, with early stopping per fold): {best_rounds}")
print(f"Best CV log-loss: {best_logloss:.4f}")

# --- 3. Fit the FINAL model on the FULL 80% training pool -------------------
#        (no data held back - this is the point of moving to 80/20 + CV)
model = xgb.XGBClassifier(
    n_estimators=best_rounds,
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train)

# --- 4. Evaluate ONCE on the held-out TEST set -------------------------------
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

print(f"\nFinal held-out TEST accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Final held-out TEST ROC-AUC: {roc_auc_score(y_test, pred_proba):.3f}")
print("\nClassification report (test set):")
print(classification_report(y_test, pred, target_names=["Stable", "Unstable"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, pred))

# --- 5. Feature importance ---------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
importances.plot.barh(ax=ax, color="#2a6f97")
ax.set_xlabel("XGBoost feature importance (gain-based)")
ax.set_title("XGBoost - key instability indicators\n(80% training pool, 5-fold CV-selected hyperparameters)")
fig.tight_layout()
fig.savefig("xgboost_feature_importance.png", dpi=150)
print("\nSaved xgboost_feature_importance.png")

# --- 6. Save the trained model for reuse -------------------------------------
model.save_model("xgboost_slope_model.json")
print("Saved trained model -> xgboost_slope_model.json")

# --- 7. Additional check: test set + the 2 real, finalised UCT cases -------
#         (test_split_with_uct.csv = test_split.csv + 2 UCT rows, tagged by
#         a 'source' column). Reported separately from step 4 above - this
#         does NOT replace the clean test-set number, it supplements it.
test_uct = pd.read_csv("test_split_with_uct.csv")
X_test_uct, y_test_uct = xy(test_uct)

pred_uct_all = model.predict(X_test_uct)
print(f"\nTest + real UCT cases ({len(test_uct)} rows) accuracy: "
      f"{accuracy_score(y_test_uct, pred_uct_all):.3f}")

uct_only = test_uct[test_uct["source"] == "UCT case data (finalised)"].copy()
X_uct, y_uct = xy(uct_only)
pred_uct = model.predict(X_uct)
pred_uct_label = np.where(pred_uct == 1, "Unstable", "Stable")

print("\nReal UCT case predictions (never seen during training or CV):")
for case_no, actual, predicted in zip(uct_only["Case No"], uct_only["Status"], pred_uct_label):
    match = "correct" if actual == predicted else "INCORRECT"
    print(f"  {case_no}: actual={actual}, predicted={predicted} ({match})")

    # Predicted probability of instability for each UCT section, and its distance from 0.5
proba_uct = model.predict_proba(X_uct)[:, 1]          # column 1 = P(Unstable)
print("\nPredicted probability of instability for the UCT sections:")
for case_no, p in zip(uct_only["Case No"], proba_uct):
    print(f"  {case_no}: P(Unstable) = {p:.4f}, margin above 0.5 = {p - 0.5:+.4f}")

Train (80%): 443, Test (20%): 111

Best hyperparameters (via 5-fold CV on the 80% training pool): max_depth=4, learning_rate=0.1
Best number of boosting rounds (from CV, with early stopping per fold): 82
Best CV log-loss: 0.3608

Final held-out TEST accuracy: 0.838
Final held-out TEST ROC-AUC: 0.912

Classification report (test set):
              precision    recall  f1-score   support

      Stable       0.82      0.85      0.84        54
    Unstable       0.85      0.82      0.84        57

    accuracy                           0.84       111
   macro avg       0.84      0.84      0.84       111
weighted avg       0.84      0.84      0.84       111

Confusion matrix (rows=actual, cols=predicted):
[[46  8]
 [10 47]]

Saved xgboost_feature_importance.png
Saved trained model -> xgboost_slope_model.json

Test + real UCT cases (113 rows) accuracy: 0.841

Real UCT case predictions (never seen during training or CV):
  UCT South side: actual=Unstable, predicted=Unstable (correct)
  UCT N

In [2]:
pd.DataFrame({"Case No": test["Case No"], "actual": y_test,
              "pred": pred, "p_unstable": pred_proba}).to_csv("test_predictions_xgboost.csv", index=False)

In [5]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["Stable", "Unstable"]
cm = confusion_matrix(y_test, pred)            # rows = actual, cols = predicted

fig, ax = plt.subplots(figsize=(5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("XGBoost confusion matrix (test set, n=111)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
fig.tight_layout()
fig.savefig("xgboost_confusion_matrix.png", dpi=300)
plt.show()

/var/folders/7s/ctpq_jjj4lx6zmyqfy7bvsjm0000gn/T/ipykernel_17869/2429992205.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["Stable", "Unstable"]
cm = confusion_matrix(y_test_uct, pred_uct_all)    # rows = actual, cols = predicted

fig, ax = plt.subplots(figsize=(5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("XGBoost confusion matrix (test set + UCT cases, n=113)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
fig.tight_layout()
fig.savefig("xgboost_confusion_matrix_with_uct.png", dpi=300)
plt.show()

/var/folders/7s/ctpq_jjj4lx6zmyqfy7bvsjm0000gn/T/ipykernel_17869/1173816494.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
"""
McNemar's exact test on the XGBoost and Random Forest test-set predictions.

Put these in the same folder and run:  python mcnemar_test.py
    test_predictions_xgboost.csv   (columns: Case No, actual, pred, p_unstable)
    test_predictions_rf.csv        (columns: Case_No, actual, pred, p_unstable)

Optional: pass the two file names on the command line
    python mcnemar_test.py xgb_file.csv rf_file.csv
Requires: pandas, scipy (pip install pandas scipy).
"""
import sys
import pandas as pd
from scipy.stats import binomtest

xgb_file = sys.argv[1] if len(sys.argv) > 2 else "test_predictions_xgboost.csv"
rf_file = sys.argv[2] if len(sys.argv) > 2 else "test_predictions_rf.csv"

x = pd.read_csv(xgb_file,encoding='latin1')
r = pd.read_csv(rf_file,encoding='latin1')

# accept either spelling of the case-number column in either file
for df in (x, r):
    df.rename(columns={c: "Case No" for c in df.columns if c.replace("_", " ").lower() == "case no"},
              inplace=True)

m = x.merge(r, on="Case No", suffixes=("_xgb", "_rf"))
if not (len(m) == len(x) == len(r)):
    raise SystemExit(f"Case lists differ: XGBoost {len(x)} rows, RF {len(r)} rows, matched {len(m)}. "
                     "Both files must come from the same test file.")
if not (m["actual_xgb"] == m["actual_rf"]).all():
    raise SystemExit("The two files disagree on the true labels for some cases; check they are the same test file.")

xgb_ok = m["pred_xgb"] == m["actual_xgb"]
rf_ok = m["pred_rf"] == m["actual_rf"]

both_ok = int((xgb_ok & rf_ok).sum())
xgb_only = int((xgb_ok & ~rf_ok).sum())   # XGBoost right, RF wrong
rf_only = int((~xgb_ok & rf_ok).sum())    # RF right, XGBoost wrong
both_wrong = int((~xgb_ok & ~rf_ok).sum())
n = len(m)

print(f"Cases: {n}")
print(f"XGBoost correct: {int(xgb_ok.sum())} ({xgb_ok.mean():.3f})   RF correct: {int(rf_ok.sum())} ({rf_ok.mean():.3f})")
print()
print("                      RF correct   RF wrong")
print(f"  XGBoost correct     {both_ok:>9d}   {xgb_only:>8d}")
print(f"  XGBoost wrong       {rf_only:>9d}   {both_wrong:>8d}")
print()
print(f"XGBoost-only correct: {xgb_only}   RF-only correct: {rf_only}   (disagreements: {xgb_only + rf_only})")
if xgb_only + rf_only == 0:
    print("The models never disagree, so the test cannot be computed (p = 1).")
else:
    p = binomtest(min(xgb_only, rf_only), xgb_only + rf_only, 0.5).pvalue
    print(f"McNemar exact (two-sided) p = {p:.4f}")
    print("p < 0.05 would indicate a difference in accuracy; otherwise the data cannot tell the models apart.")

Cases: 111
XGBoost correct: 93 (0.838)   RF correct: 97 (0.874)

                      RF correct   RF wrong
  XGBoost correct            92          1
  XGBoost wrong               5         13

XGBoost-only correct: 1   RF-only correct: 5   (disagreements: 6)
McNemar exact (two-sided) p = 0.2188
p < 0.05 would indicate a difference in accuracy; otherwise the data cannot tell the models apart.


In [11]:
"""
XGBoost feature importance that measures PREDICTIVE performance.

Gain (model.feature_importances_) = how much a feature reduces the training loss
inside the trees. Permutation importance = how much the held-out test accuracy
(and AUC) drops when one feature's values are shuffled, which is the quantity
the report text should talk about when it says "prediction accuracy".

Run in the folder with train80_split.csv and test_split.csv:  python xgboost_permutation_importance.py
Requires: pandas, numpy, xgboost, scikit-learn, matplotlib
"""
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURES = ["unit_weight_kNm3", "cohesion_kPa", "friction_angle_deg",
            "slope_angle_deg", "slope_height_m", "pore_pressure_ratio"]
LABELS = {"unit_weight_kNm3": "Unit weight", "cohesion_kPa": "Cohesion",
          "friction_angle_deg": "Friction angle", "slope_angle_deg": "Slope angle",
          "slope_height_m": "Slope height", "pore_pressure_ratio": "Pore pressure ratio"}

train = pd.read_csv("train80_split.csv"); test = pd.read_csv("test_split.csv")
xy = lambda d: (d[FEATURES].values, (d["Status"] == "Unstable").astype(int).values)
X_train, y_train = xy(train); X_test, y_test = xy(test)

# Same settings as the reported model (depth 4, lr 0.1, 82 rounds, 0.8 subsampling, seed 42)
model = xgb.XGBClassifier(n_estimators=82, max_depth=4, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8,
                          eval_metric="logloss", random_state=42).fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Check: test accuracy = {acc:.3f} (reported 0.838; if different, the refit does not match the reported model)")

res = permutation_importance(model, X_test, y_test,
                             scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
                             n_repeats=50, random_state=42)
out = pd.DataFrame({
    "feature": FEATURES,
    "gain": model.feature_importances_,
    "acc_drop_mean": res["accuracy"].importances_mean,
    "acc_drop_sd": res["accuracy"].importances_std,
    "auc_drop_mean": res["roc_auc"].importances_mean,
    "auc_drop_sd": res["roc_auc"].importances_std,
}).sort_values("acc_drop_mean", ascending=False)
print(out.round(4).to_string(index=False))
out.to_csv("xgboost_importance_gain_vs_permutation.csv", index=False)

o = out.sort_values("acc_drop_mean")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh([LABELS[f] for f in o.feature], o.acc_drop_mean, xerr=o.acc_drop_sd, color="#2a6f97", capsize=3)
ax.set_xlabel("Decrease in test accuracy when the feature is permuted")
ax.set_ylabel("Predictor")
ax.axvline(0, color="grey", lw=0.8)
fig.savefig("xgboost_permutation_importance.png", dpi=300, bbox_inches="tight")
print("Saved xgboost_permutation_importance.png and xgboost_importance_gain_vs_permutation.csv")

Check: test accuracy = 0.838 (reported 0.838; if different, the refit does not match the reported model)
            feature   gain  acc_drop_mean  acc_drop_sd  auc_drop_mean  auc_drop_sd
       cohesion_kPa 0.1871         0.1068       0.0285         0.1016       0.0239
 friction_angle_deg 0.1897         0.0589       0.0244         0.0530       0.0153
pore_pressure_ratio 0.1077         0.0341       0.0182         0.0221       0.0116
   unit_weight_kNm3 0.2005         0.0339       0.0152         0.0414       0.0123
     slope_height_m 0.1697         0.0285       0.0211         0.0351       0.0111
    slope_angle_deg 0.1454         0.0263       0.0199         0.0477       0.0149
Saved xgboost_permutation_importance.png and xgboost_importance_gain_vs_permutation.csv
